In [1]:
from kfunc_filter import should_filter_function
from ollama import chat
from collections import defaultdict, deque
import networkx as nx
import matplotlib.pyplot as plt
import os
import scipy
import re
import time

start_time = time.perf_counter()
model = 'qwen3:32b'
role = 'user'
num_ctx = 14336
the_directory = 'output/demo'
demo_profile = "./demo-profile-sysclose.txt"
search_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
kernel_cg_path = "./pruned_callgraph.graphml"
k_cg = nx.read_graphml(kernel_cg_path)
k_cg.remove_edges_from(nx.selfloop_edges(k_cg))
kall_functions = [node for node in k_cg.nodes if not should_filter_function(node)]

def gen_subgraph(static_graph, sys_entry_function, function_set, remove_broken_edges=False):
    function_set = [f for f in function_set if not should_filter_function(f)]
    subgraph_nodes = set(function_set)
    if remove_broken_edges:
        reachable_nodes = nx.descendants(static_graph, sys_entry_function)
        reachable_nodes.add(sys_entry_function)
        subgraph_nodes = subgraph_nodes.intersection(reachable_nodes)
    subgraph = static_graph.subgraph(subgraph_nodes).copy()
    return subgraph

def outedges_callgraph(graph, function_name, do_filter=True):
    if function_name not in graph:
        print(f"Function {function_name} does not exist in the graph.")
        return []
    out_edges = []
    for _, target, edge_data in graph.out_edges(function_name, data=True):
        if do_filter and should_filter_function(target):
            continue
        out_edges.append((function_name, target, edge_data))
    return out_edges

def export_graph_as_markdown_tree(graph):
    def dfs(node, indent=0, visited=set()):
        lines = []
        prefix = "  " * indent + "- " + node
        lines.append(prefix)
        visited.add(node)
        for _, neighbor in graph.out_edges(node):
            if neighbor not in visited:
                lines.extend(dfs(neighbor, indent + 1, visited))
        return lines
    roots = [n for n in graph.nodes if graph.in_degree(n) == 0]
    all_lines = []
    for root in roots:
        all_lines.extend(dfs(root, indent=0, visited=set()))
    return "\n".join(all_lines)

def find_function_source(root_dir, function_name, exts=('.c',)):
    sig_re = re.compile(
        rf'''^[ \t]*
            (?:[A-Za-z_]\w*  # return type or storage-class (e.g. "static", "inline", "void", "struct foo")
               (?:\s+[\w\*\s]+)*)?  # allow pointers and multiple words
            \b{re.escape(function_name)}\s*\(  # the function name
        ''',
        re.VERBOSE
    )
    def scan_file(path):
        try:
            f = open(path, 'r', errors='ignore')
        except PermissionError:
            return None
        with f:
            collecting = False
            brace_count = 0
            buffer = []
            in_block = False
            for raw in f:
                line = raw
                if in_block:
                    end = line.find('*/')
                    if end >= 0:
                        in_block = False
                        line = line[end+2:]
                    else:
                        continue
                start = line.find('/*')
                if start >= 0:
                    end = line.find('*/', start+2)
                    if end >= 0:
                        line = line[:start] + line[end+2:]
                    else:
                        in_block = True
                        line = line[:start]
                clean = line.split('//', 1)[0]
                if clean.rstrip().endswith('\\'):
                    continue
                if not collecting:
                    if sig_re.match(clean):
                        sig_raw = [raw]
                        sig_clean = [clean]
                        bal = clean.count('(') - clean.count(')')
                        while bal > 0:
                            nxt_raw = f.readline()
                            if not nxt_raw:
                                break
                            nxt = nxt_raw
                            if in_block:
                                end = nxt.find('*/')
                                if end >= 0:
                                    in_block = False
                                    nxt = nxt[end+2:]
                                else:
                                    continue
                            st = nxt.find('/*')
                            if st >= 0:
                                ed = nxt.find('*/', st+2)
                                if ed >= 0:
                                    nxt = nxt[:st] + nxt[ed+2:]
                                else:
                                    in_block = True
                                    nxt = nxt[:st]
                            nxt_clean = nxt.split('//',1)[0]
                            sig_raw.append(nxt_raw)
                            sig_clean.append(nxt_clean)
                            bal += nxt_clean.count('(') - nxt_clean.count(')')
                        if ''.join(sig_clean).strip().endswith(';'):
                            continue
                        buffer = sig_raw.copy()
                        if '{' in sig_clean[-1]:
                            collecting = True
                            brace_count = sig_clean[-1].count('{') - sig_clean[-1].count('}')
                        else:
                            for body_raw in f:
                                body = body_raw
                                if in_block:
                                    end = body.find('*/')
                                    if end >= 0:
                                        in_block = False
                                        body = body[end+2:]
                                    else:
                                        continue
                                st = body.find('/*')
                                if st >= 0:
                                    ed = body.find('*/', st+2)
                                    if ed >= 0:
                                        body = body[:st] + body[ed+2:]
                                    else:
                                        in_block = True
                                        body = body[:st]
                                body_clean = body.split('//',1)[0]
                                buffer.append(body_raw)
                                if '{' in body_clean:
                                    collecting = True
                                    brace_count = (
                                        body_clean.count('{') - 
                                        body_clean.count('}')
                                    )
                                    break
                        if not collecting:
                            buffer = []
                else:
                    buffer.append(raw)
                    tmp = raw
                    if in_block:
                        end = tmp.find('*/')
                        if end >= 0:
                            in_block = False
                            tmp = tmp[end+2:]
                        else:
                            continue
                    st = tmp.find('/*')
                    if st >= 0:
                        ed = tmp.find('*/', st+2)
                        if ed >= 0:
                            tmp = tmp[:st] + tmp[ed+2:]
                        else:
                            in_block = True
                            tmp = tmp[:st]
                    tmp_clean = tmp.split('//',1)[0]

                    brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                    if brace_count == 0:
                        return ''.join(buffer)
        return None
    def recurse(path):
        try:
            entries = os.scandir(path)
        except PermissionError:
            return None
        with entries:
            for e in entries:
                name = e.name.lower()
                if name.endswith(exts):
                    try:
                        if e.is_file(follow_symlinks=False):
                            src = scan_file(e.path)
                            if src:
                                return src
                    except PermissionError:
                        pass
                try:
                    if e.is_dir(follow_symlinks=False):
                        found = recurse(e.path)
                        if found:
                            return found
                except PermissionError:
                    pass
        return None
    return recurse(root_dir)

def find_call_chain(src, dst, calls):
    visited = set([src])
    queue = deque([[src]])
    while queue:
        path = queue.popleft()
        last = path[-1]
        if last == dst:
            return path
        for callee in calls.get(last, []):
            if callee not in visited:
                visited.add(callee)
                queue.append(path + [callee])
    return None

def get_in_between_functions(calls, src, dst):
    chain = find_call_chain(src, dst, calls)
    if chain and len(chain) > 2:
        return chain[1:-1]
    return []

demo_profiled_functions = set()

with open(demo_profile, "r") as f:
    lines = f.readlines()
    for line in lines:
        func_name = line.strip()
        if func_name in kall_functions and not should_filter_function(func_name):
            demo_profiled_functions.add(func_name)

subgraph = gen_subgraph(k_cg, "__x64_sys_close", demo_profiled_functions)
subgraph_start = subgraph.copy()

print('Done!')

Done!


In [2]:
is_there_something_new = True
has_been_checked = []

while (is_there_something_new):
    is_there_something_new = False
    topo_sorted = list(nx.topological_sort(subgraph_start))
    reverse_topo_sorted = list(reversed(topo_sorted))
    try_this = defaultdict(list)
    for i in reverse_topo_sorted:
        out_edges = outedges_callgraph(k_cg, i)
        for src, dst, edge_data in out_edges:
            try_this[src].append(dst)
    markdown_output = export_graph_as_markdown_tree(subgraph_start)
    for src, targets in try_this.items():
        found_function_code = 'Some function code.'
        for dst in targets:
            if subgraph_start.has_edge(src, dst):
                continue
            if dst in has_been_checked:
                continue
            else:
                has_been_checked.append(dst)
            is_there_something_new = True
            subgraph_start.add_edge(src, dst, inferred=True)

markdown_output = export_graph_as_markdown_tree(subgraph_start)
the_markdown = f'{the_directory}/the_start_markdown.txt'

with open(the_markdown, 'w', encoding='utf-8') as file:
    file.write(markdown_output)

print('Done!')

Done!


In [8]:
import subprocess
import json
import shlex
from pathlib import Path

def find_function_native(root_dir: str, function_name: str) -> str | None:
    """
    Uses grep+awk to extract the first C definition of `function_name` under root_dir.
    Returns the full 'name(...) { ... }' text, or None if not found.
    """
    # Step 1: grep for the first .c file containing a candidate definition
    try:
        # -n to get line numbers, head -n1 to pick the first hit
        grep_cmd = [
            "grep", "-R", "-n", "--include=*.c",
            f"^[[:space:]]*{function_name}[[:space:]]*\\(",
            root_dir
        ]
        out = subprocess.check_output(grep_cmd, text=True)
    except subprocess.CalledProcessError:
        return None

    first_hit = out.splitlines()[0]
    file_path = first_hit.split(":", 1)[0]

    # Step 2: use awk to strip comments, match at start-of-line, balance braces
    awk_script = r'''
BEGIN {
  pat = "^[[:space:]]*" fname "[[:space:]]*\\("
  in_func = 0
  brace = 0
}
{
  # strip // comments
  line = $0
  sub(/\/\/.*/, "", line)
  # strip simple /* ... */ on the same line
  gsub(/\/\*[^*]*\*\//, "", line)
  # skip macro-continuations
  if (line ~ /\\[[:space:]]*$/) next

  if (!in_func) {
    if (line ~ pat) {
      in_func = 1
      buf = $0 "\n"
      brace += gsub(/\{/, "{") - gsub(/\}/, "}")
    }
  }
  else {
    buf = buf $0 "\n"
    brace += gsub(/\{/, "{") - gsub(/\}/, "}")
    if (brace == 0) {
      printf "%s", buf
      exit
    }
  }
}
'''
    try:
        awk_cmd = [
            "awk",
            "-v", f"fname={function_name}",
            awk_script,
            file_path
        ]
        code = subprocess.check_output(awk_cmd, text=True)
        return code.strip() or None
    except subprocess.CalledProcessError:
        return None

In [ ]:
# 1) Set your root directory
# 2) List of functions you want to extract
function_list = [
    "fput",
    "fsnotify_find_mark",
    "kfree",
    # ... etc
]

# 3) Build your results dict
results: dict[str, str | None] = {}
for fn in function_list:
    src = find_function_native(subgraph_start, fn)
    results[fn] = src

# 4) Write out to JSON file
the_directory = subgraph_start      # or any other folder you choose
Path(the_directory).mkdir(parents=True, exist_ok=True)
file_path = f"{the_directory}/the_functions_all.txt"
with open(file_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Wrote {len(results)} functions to {file_path}")

In [17]:
import subprocess
import re
from pathlib import Path

def find_function_source_in_file(path: str, function_name: str) -> str | None:
    """
    Scan a single .c file at `path` for the real definition of `function_name`.
    Returns the full 'signature + body' if found, or None otherwise.
    """
    sig_re = re.compile(rf'\b{re.escape(function_name)}\b\s*\(')

    try:
        f = open(path, 'r', errors='ignore')
    except PermissionError:
        return None

    with f:
        collecting = False
        brace_count = 0
        buffer = []
        in_block = False

        for raw in f:
            line = raw

            # —— Strip block comments (stateful) ——
            if in_block:
                end = line.find('*/')
                if end >= 0:
                    in_block = False
                    line = line[end+2:]
                else:
                    continue

            start = line.find('/*')
            if start >= 0:
                end = line.find('*/', start+2)
                if end >= 0:
                    line = line[:start] + line[end+2:]
                else:
                    in_block = True
                    line = line[:start]

            # —— Strip // comments ——
            clean = line.split('//', 1)[0]

            # —— Skip macro continuations ——
            if clean.rstrip().endswith('\\'):
                continue

            if not collecting:
                # Look for name( anywhere in the cleaned line
                if sig_re.search(clean):
                    # Gather raw lines until parentheses balance
                    sig_raw = [raw]
                    sig_clean = [clean]
                    bal = clean.count('(') - clean.count(')')
                    while bal > 0:
                        nxt_raw = f.readline()
                        if not nxt_raw:
                            break
                        # strip comments for balance
                        nxt = nxt_raw
                        if in_block:
                            end = nxt.find('*/')
                            if end >= 0:
                                in_block = False
                                nxt = nxt[end+2:]
                            else:
                                continue
                        st = nxt.find('/*')
                        if st >= 0:
                            ed = nxt.find('*/', st+2)
                            if ed >= 0:
                                nxt = nxt[:st] + nxt[ed+2:]
                            else:
                                in_block = True
                                nxt = nxt[:st]
                        nxt_clean = nxt.split('//',1)[0]

                        sig_raw.append(nxt_raw)
                        sig_clean.append(nxt_clean)
                        bal += nxt_clean.count('(') - nxt_clean.count(')')

                    # If it ends with ';' (prototype or call), skip it
                    if ''.join(sig_clean).strip().endswith(';'):
                        continue

                    # Start collecting the body
                    buffer = sig_raw.copy()
                    if '{' in sig_clean[-1]:
                        collecting = True
                        brace_count = sig_clean[-1].count('{') - sig_clean[-1].count('}')
                    else:
                        for body_raw in f:
                            # strip comments for brace count
                            body = body_raw
                            if in_block:
                                end = body.find('*/')
                                if end >= 0:
                                    in_block = False
                                    body = body[end+2:]
                                else:
                                    continue
                            st = body.find('/*')
                            if st >= 0:
                                ed = body.find('*/', st+2)
                                if ed >= 0:
                                    body = body[:st] + body[ed+2:]
                                else:
                                    in_block = True
                                    body = body[:st]
                            body_clean = body.split('//',1)[0]

                            buffer.append(body_raw)
                            if '{' in body_clean:
                                collecting = True
                                brace_count = body_clean.count('{') - body_clean.count('}')
                                break

                    # Bail out if no '{' ever found
                    if not collecting:
                        buffer = []
            else:
                # Inside the body: keep going until braces balance
                buffer.append(raw)
                tmp = raw
                if in_block:
                    end = tmp.find('*/')
                    if end >= 0:
                        in_block = False
                        tmp = tmp[end+2:]
                    else:
                        continue
                st = tmp.find('/*')
                if st >= 0:
                    ed = tmp.find('*/', st+2)
                    if ed >= 0:
                        tmp = tmp[:st] + tmp[ed+2:]
                    else:
                        in_block = True
                        tmp = tmp[:st]
                tmp_clean = tmp.split('//',1)[0]

                brace_count += tmp_clean.count('{') - tmp_clean.count('}')
                if brace_count == 0:
                    return ''.join(buffer)

    return None


def find_function_native(root_dir: str, function_name: str) -> str | None:
    """
    1) Grep (in C) for any .c file that mentions function_name(.
    2) For each candidate, call find_function_source_in_file()
       until we get a real definition.
    """
    # use -E for ERE, or -P for PCRE. Here we use -E and [[:space:]] to match whitespace.
    pat = rf'\b{re.escape(function_name)}\s*\('
    try:
        out = subprocess.check_output([
            'grep', '-R', '-lE',
            '--include=*.c',
            pat,
            root_dir
        ], text=True)
    except subprocess.CalledProcessError:
        return None

    for file_path in out.splitlines():
        src = find_function_source_in_file(file_path, function_name)
        if src:
            return src

    return None

In [14]:
search_dir

'/home/wensan/chuqi_4/cleanslate-kerntracer'

In [ ]:
the_dictionary_functions = {}
file_path = f"{the_directory}/the_functions_all.txt"

with open(file_path, "a") as f:
    for node in subgraph_start.nodes():
        print("SOURCE CODE:", node)
        src = find_function_native(search_dir, node)
        print(src)
        print()
        print()
        print()
        print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
        print()
        print()
        print()
        if src is not None:
            the_dictionary_functions[node] = src
            f.write(f"Source Code for {node}:\n")
            f.write(src)
            f.write("\n\n")

SOURCE CODE: fsnotify_find_mark
struct fsnotify_mark *fsnotify_find_mark(fsnotify_connp_t *connp,
					 struct fsnotify_group *group)
{
	struct fsnotify_mark_connector *conn;
	struct fsnotify_mark *mark;

	conn = fsnotify_grab_connector(connp);
	if (!conn)
		return NULL;

	hlist_for_each_entry(mark, &conn->list, obj_list) {
		if (mark->group == group &&
		    (mark->flags & FSNOTIFY_MARK_FLAG_ATTACHED)) {
			fsnotify_get_mark(mark);
			spin_unlock(&conn->lock);
			return mark;
		}
	}
	spin_unlock(&conn->lock);
	return NULL;
}




XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX



SOURCE CODE: fput
void fput(struct file *file)
{
	if (atomic_long_dec_and_test(&file->f_count)) {
		struct task_struct *task = current;

		if (likely(!in_interrupt() && !(task->flags & PF_KTHREAD))) {
			init_task_work(&file->f_rcuhead, ____fput);
			if (!task_work_add(task, &file->f_rcuhead, TWA_RESUME))
				return;
			/*
			 */
		

In [3]:
is_there_something_new = True
has_been_checked = []
the_output = f'{the_directory}/the_real_llm.txt'

with open(the_output, 'w', encoding='utf-8') as f:
    f.write('')

function_sources = {}
node_set = set(subgraph.nodes)
start = True

while (is_there_something_new):
    new_nodes = list(set(subgraph.nodes) - node_set)
    if start:
        new_nodes = list(node_set)
        start = False
    for node in new_nodes:
        found_function_code = find_function_source(search_dir, node)
        function_sources[node] = found_function_code
    is_there_something_new = False
    topo_sorted = list(nx.topological_sort(subgraph))
    reverse_topo_sorted = list(reversed(topo_sorted))
    try_this = defaultdict(list)
    for i in reverse_topo_sorted:
        out_edges = outedges_callgraph(k_cg, i)
        for src, dst, edge_data in out_edges:
            try_this[src].append(dst)
    markdown_output = export_graph_as_markdown_tree(subgraph)
    for src, targets in try_this.items():
        for dst in targets:
            if subgraph.has_edge(src, dst):
                continue
            if dst in has_been_checked:
                continue
            else:
                has_been_checked.append(dst)
            dst_code = find_function_source(search_dir, dst)
            prompt_text = f'''You are a Linux security expert analyzing kernel call-graph edges. The following historical dynamic execution path was observed:
{markdown_output}
Source code for {dst}:
{dst_code}
'''
            for node, code in function_sources.items():
                prompt_text += f"Source code for {node}:\n{code}"
            in_between_functions = get_in_between_functions(k_cg, src, dst)
            for fn in in_between_functions:
                code = find_function_source(search_dir, fn)
                prompt_text += f"Source code for intermediate {fn}:\n{code}"
            prompt_text += f'From a security-engineering standpoint, is it reasonable to expect that execution of {src} will reach {dst}? Provide a concise technical justification for your position, then state your conclusion in exactly this format: {{Your justification}}\\nFINAL ANSWER -> YES/NO'
            prompt = {
                'role': 'user',
                'content': prompt_text
            }
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write(f'🟢🟢🟢PROMPT🟢🟢🟢: {prompt_text}\n')
            response = chat(model=model, messages=[{'role': role, 'content': prompt_text}], options={'num_ctx': num_ctx})
            filtered_response = re.sub(r'<think>.*?</think>', '', response.message.content, flags=re.DOTALL).strip()
            filtered_response = re.sub(r'[ \t]+$', '', filtered_response, flags=re.MULTILINE)
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write(f'🟢🟢🟢RESPONSE🟢🟢🟢: {filtered_response}\n')
            if "FINAL ANSWER -> YES" in filtered_response.upper():
                is_there_something_new = True
                with open(the_output, 'a', encoding='utf-8') as file:
                    file.write(f'🟢 Upgrading edge {src} -> {dst} to subgraph (LLM confirmed)')
                    file.write('\n')
                subgraph.add_edge(src, dst, inferred=True)
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write('\n\n\n')
    if is_there_something_new:
        with open(the_output, 'a', encoding='utf-8') as file:
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('\n\n\n\n')
            file.write('\n')

with open(the_output, 'a', encoding='utf-8') as file:
    file.write('THE END!')

print('Done!')


KeyboardInterrupt



In [ ]:
markdown_output = export_graph_as_markdown_tree(subgraph)
the_markdown = f'{the_directory}/the_real_markdown.txt'

with open(the_markdown, 'w', encoding='utf-8') as file:
    file.write(markdown_output)

print('Done!')

In [ ]:
end_time = time.perf_counter()
elapsed = end_time - start_time
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

print(f"Elapsed time: {hours}h {minutes}m {seconds:.6f}s")
print('Done!')

In [ ]:
start_time = time.perf_counter()
the_directory = 'output/nginx'
demo_profile = "./nginx-ltp-redis"

demo_profiled_functions = set()

with open(demo_profile, "r") as f:
    lines = f.readlines()
    for line in lines:
        func_name = line.strip()
        if func_name in kall_functions and not should_filter_function(func_name):
            demo_profiled_functions.add(func_name)

subgraph = gen_subgraph(k_cg, "__x64_sys_close", demo_profiled_functions)
subgraph_start = subgraph.copy()

##################################################
is_there_something_new = True
has_been_checked = []

while (is_there_something_new):
    is_there_something_new = False
    topo_sorted = list(nx.topological_sort(subgraph_start))
    reverse_topo_sorted = list(reversed(topo_sorted))
    try_this = defaultdict(list)
    for i in reverse_topo_sorted:
        out_edges = outedges_callgraph(k_cg, i)
        for src, dst, edge_data in out_edges:
            try_this[src].append(dst)
    markdown_output = export_graph_as_markdown_tree(subgraph_start)
    for src, targets in try_this.items():
        found_function_code = 'Some function code.'
        for dst in targets:
            if subgraph_start.has_edge(src, dst):
                continue
            if dst in has_been_checked:
                continue
            else:
                has_been_checked.append(dst)
            is_there_something_new = True
            subgraph_start.add_edge(src, dst, inferred=True)

markdown_output = export_graph_as_markdown_tree(subgraph_start)
the_markdown = f'{the_directory}/the_start_markdown.txt'

with open(the_markdown, 'w', encoding='utf-8') as file:
    file.write(markdown_output)

##################################################
is_there_something_new = True
has_been_checked = []
the_output = f'{the_directory}/the_real_llm.txt'

with open(the_output, 'w', encoding='utf-8') as f:
    f.write('')

function_sources = {}
node_set = set(subgraph.nodes)
start = True

while (is_there_something_new):
    new_nodes = list(set(subgraph.nodes) - node_set)
    if start:
        new_nodes = list(node_set)
        start = False
    for node in new_nodes:
        found_function_code = find_function_source(search_dir, node)
        function_sources[node] = found_function_code
    is_there_something_new = False
    topo_sorted = list(nx.topological_sort(subgraph))
    reverse_topo_sorted = list(reversed(topo_sorted))
    try_this = defaultdict(list)
    for i in reverse_topo_sorted:
        out_edges = outedges_callgraph(k_cg, i)
        for src, dst, edge_data in out_edges:
            try_this[src].append(dst)
    markdown_output = export_graph_as_markdown_tree(subgraph)
    for src, targets in try_this.items():
        for dst in targets:
            if subgraph.has_edge(src, dst):
                continue
            if dst in has_been_checked:
                continue
            else:
                has_been_checked.append(dst)
            dst_code = find_function_source(search_dir, dst)
            prompt_text = f'''You are a Linux security expert analyzing kernel call-graph edges. The following historical dynamic execution path was observed:
{markdown_output}
Source code for {dst}:
{dst_code}
'''
            for node, code in function_sources.items():
                prompt_text += f"Source code for {node}:\n{code}"
            in_between_functions = get_in_between_functions(k_cg, src, dst)
            for fn in in_between_functions:
                code = find_function_source(search_dir, fn)
                prompt_text += f"Source code for intermediate {fn}:\n{code}"
            prompt_text += f'From a security-engineering standpoint, is it reasonable to expect that execution of {src} will reach {dst}? Provide a concise technical justification for your position, then state your conclusion in exactly this format: {{Your justification}}\\nFINAL ANSWER -> YES/NO'
            prompt = {
                'role': 'user',
                'content': prompt_text
            }
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write(f'🟢🟢🟢PROMPT🟢🟢🟢: {prompt_text}\n')
            response = chat(model=model, messages=[{'role': role, 'content': prompt_text}], options={'num_ctx': num_ctx})
            filtered_response = re.sub(r'<think>.*?</think>', '', response.message.content, flags=re.DOTALL).strip()
            filtered_response = re.sub(r'[ \t]+$', '', filtered_response, flags=re.MULTILINE)
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write(f'🟢🟢🟢RESPONSE🟢🟢🟢: {filtered_response}\n')
            if "FINAL ANSWER -> YES" in filtered_response.upper():
                is_there_something_new = True
                with open(the_output, 'a', encoding='utf-8') as file:
                    file.write(f'🟢 Upgrading edge {src} -> {dst} to subgraph (LLM confirmed)')
                    file.write('\n')
                subgraph.add_edge(src, dst, inferred=True)
            with open(the_output, 'a', encoding='utf-8') as file:
                file.write('\n\n\n')
    if is_there_something_new:
        with open(the_output, 'a', encoding='utf-8') as file:
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('🔴🔴🔴NEW SUBGRAPH!🔴🔴🔴')
            file.write('\n')
            file.write('\n\n\n\n')
            file.write('\n')

with open(the_output, 'a', encoding='utf-8') as file:
    file.write('THE END!')

##################################################
markdown_output = export_graph_as_markdown_tree(subgraph)
the_markdown = f'{the_directory}/the_real_markdown.txt'

with open(the_markdown, 'w', encoding='utf-8') as file:
    file.write(markdown_output)

##################################################
end_time = time.perf_counter()
elapsed = end_time - start_time
hours   = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = elapsed % 60

print(f"Elapsed time: {hours}h {minutes}m {seconds:.6f}s")
print('Done!')